In [1]:
# If needed, uncomment:
# !pip -q install earthengine-api tqdm pandas numpy

In [2]:
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import ee
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [3]:
# Run these once in Colab
import certifi, os
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["CURL_CA_BUNDLE"] = certifi.where()
ee.Authenticate()

# Recommended: pass your GCP project id
# ee.Initialize(project='YOUR_GCP_PROJECT_ID')

# If you already authenticated in the runtime, this may be enough:
ee.Initialize()

In [4]:
TRAIN_PATH = os.path.join(PROJECT_ROOT, "Datasets_Provided", "water_quality_training_dataset.csv")
VALID_PATH = os.path.join(PROJECT_ROOT, "submission_template.csv")

OUT_TRAIN_PATH = os.path.join(PROJECT_ROOT, "Datasets_Ours", "gaia_features_training.csv")
OUT_VALID_PATH = os.path.join(PROJECT_ROOT, "Datasets_Ours", "gaia_features_validation.csv")

BUFFER_M = 120   # try 90, 120, 300 later
SCALE_M = 30     # GAIA native resolution is 30m
CHUNK_SIZE = 200
RESUME = True    # set False to overwrite existing outputs

GAIA_IMG = ee.Image("Tsinghua/FROM-GLC/GAIA/v10").select("change_year_index")

In [ ]:
# GAIA v2 helpers (use these for extraction)

def infer_column(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    for c in df.columns:
        lc = c.lower()
        if any(k in lc for k in candidates):
            return c
    return None


def count_rows_in_csv(path: str) -> int:
    with open(path, 'r', encoding='utf-8') as f:
        return max(sum(1 for _ in f) - 1, 0)


def parse_year(x):
    if pd.isna(x):
        return np.nan
    dt = pd.to_datetime(x, errors='coerce', dayfirst=True)
    if pd.isna(dt):
        return np.nan
    return int(dt.year)


def expected_output_columns():
    return [
        "Latitude",
        "Longitude",
        "Sample Date",
        "gaia_changed_ever_frac",
        "gaia_impervious_frac_by_sample_year",
        "gaia_recent_change_5y_frac",
        "gaia_years_since_change_mean",
        "gaia_transition_year_mean_changed_pixels",
    ]


def build_gaia_feature_image_v2(sample_year):
    idx = GAIA_IMG

    transition_year = idx.expression(
        "(i > 0) ? (2019 - i) : 0",
        {"i": idx}
    ).rename("gaia_transition_year")

    # Use unmask to avoid nulls in non-impervious areas
    changed_ever = idx.gt(0).rename("gaia_changed_ever").unmask(0)
    impervious_by_year = idx.gt(0).And(transition_year.lte(sample_year)).rename("gaia_impervious_by_year").unmask(0)
    recent_change_5y = idx.gt(0).And(transition_year.gte(sample_year - 4)).And(
        transition_year.lte(sample_year)
    ).rename("gaia_recent_change_5y").unmask(0)

    years_since_change = ee.Image.constant(sample_year).subtract(transition_year).updateMask(
        idx.gt(0).And(transition_year.lte(sample_year))
    ).rename("gaia_years_since_change").unmask(-1)

    transition_year_changed = transition_year.updateMask(idx.gt(0)).rename("gaia_transition_year_changed").unmask(-1)

    return ee.Image.cat([
        changed_ever,
        impervious_by_year,
        recent_change_5y,
        years_since_change,
        transition_year_changed
    ])


def compute_gaia_stats_v2(lat, lon, sample_year, buffer_m=120, scale=30, retries=3):
    if pd.isna(lat) or pd.isna(lon) or pd.isna(sample_year):
        return {
            "gaia_changed_ever_frac": np.nan,
            "gaia_impervious_frac_by_sample_year": np.nan,
            "gaia_recent_change_5y_frac": np.nan,
            "gaia_years_since_change_mean": np.nan,
            "gaia_transition_year_mean_changed_pixels": np.nan,
        }

    geom = ee.Geometry.Point([float(lon), float(lat)]).buffer(float(buffer_m))
    img = build_gaia_feature_image_v2(int(sample_year))

    out = None
    for attempt in range(retries):
        try:
            out = img.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=geom,
                scale=scale,
                maxPixels=1e9,
                bestEffort=True
            ).getInfo()
            break
        except Exception as e:
            if attempt == retries - 1:
                print(f"EE error at lat={lat}, lon={lon}, year={sample_year}: {e}")
                out = {}
            else:
                time.sleep(1.5 * (attempt + 1))

    return {
        "gaia_changed_ever_frac": out.get("gaia_changed_ever"),
        "gaia_impervious_frac_by_sample_year": out.get("gaia_impervious_by_year"),
        "gaia_recent_change_5y_frac": out.get("gaia_recent_change_5y"),
        "gaia_years_since_change_mean": out.get("gaia_years_since_change"),
        "gaia_transition_year_mean_changed_pixels": out.get("gaia_transition_year_changed"),
    }


def add_gaia_features_v2(df, lat_col, lon_col, date_col, buffer_m=120, scale=30):
    work = df.copy().reset_index(drop=True)

    # Normalize empty strings to NaN for critical columns
    work[[lat_col, lon_col, date_col]] = (
        work[[lat_col, lon_col, date_col]]
        .replace(r"^\s*$", np.nan, regex=True)
    )

    work["_sample_year"] = work[date_col].apply(parse_year)

    cache = {}
    feature_rows = []

    for _, row in tqdm(work.iterrows(), total=len(work), desc="Extracting GAIA features"):
        key = (
            round(float(row[lat_col]), 6) if pd.notna(row[lat_col]) else None,
            round(float(row[lon_col]), 6) if pd.notna(row[lon_col]) else None,
            int(row["_sample_year"]) if pd.notna(row["_sample_year"]) else None,
            buffer_m,
            scale
        )
        if key not in cache:
            cache[key] = compute_gaia_stats_v2(
                lat=row[lat_col],
                lon=row[lon_col],
                sample_year=row["_sample_year"],
                buffer_m=buffer_m,
                scale=scale
            )
        feature_rows.append(cache[key])

    feat_df = pd.DataFrame(feature_rows)
    base_df = pd.DataFrame({
        "Latitude": pd.to_numeric(work[lat_col], errors="coerce"),
        "Longitude": pd.to_numeric(work[lon_col], errors="coerce"),
        "Sample Date": work[date_col],
    })

    return pd.concat([base_df, feat_df], axis=1)


In [6]:
train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VALID_PATH)

lat_col_train = infer_column(train_df, ["latitude", "lat"])
lon_col_train = infer_column(train_df, ["longitude", "lon", "lng"])
date_col_train = infer_column(train_df, ["sample date", "sample_date", "date", "datetime"])

lat_col_valid = infer_column(valid_df, ["latitude", "lat"])
lon_col_valid = infer_column(valid_df, ["longitude", "lon", "lng"])
date_col_valid = infer_column(valid_df, ["sample date", "sample_date", "date", "datetime"])

print("Train columns:", lat_col_train, lon_col_train, date_col_train)
print("Valid columns:", lat_col_valid, lon_col_valid, date_col_valid)

if None in [lat_col_train, lon_col_train, date_col_train]:
    raise ValueError("Could not find required train columns for latitude, longitude, and sample date.")
if None in [lat_col_valid, lon_col_valid, date_col_valid]:
    raise ValueError("Could not find required validation columns for latitude, longitude, and sample date.")

Train columns: Latitude Longitude Sample Date
Valid columns: Latitude Longitude Sample Date


In [7]:
def extract_chunked(df, out_path, lat_col, lon_col, date_col, label):
    start_idx = 0
    expected_cols = expected_output_columns()

    if os.path.exists(out_path):
        if not RESUME:
            os.remove(out_path)
        else:
            existing_cols = list(pd.read_csv(out_path, nrows=0).columns)
            if existing_cols != expected_cols:
                raise ValueError(
                    "Existing output schema does not match the current extractor. "
                    "Delete the output CSV or set RESUME=False to overwrite."
                )
            existing_check = pd.read_csv(out_path, usecols=["Latitude", "Longitude", "Sample Date"])
            if existing_check.isna().any().any():
                raise ValueError(
                    "Existing output contains blank Latitude/Longitude/Sample Date. "
                    "Delete the output CSV or set RESUME=False to overwrite."
                )
            start_idx = count_rows_in_csv(out_path)

    print(f"🚀 Running GAIA extraction for {label} (chunked)...")
    print(f"Total rows: {len(df)}")
    print(f"Output file: {out_path}")
    print(f"Chunk size: {CHUNK_SIZE}")
    print(f"Resuming from row index: {start_idx}")

    for chunk_start in range(start_idx, len(df), CHUNK_SIZE):
        chunk_end = min(chunk_start + CHUNK_SIZE, len(df))
        chunk_df = df.iloc[chunk_start:chunk_end].copy()

        # Normalize empty strings to NaN before checks
        chunk_df[[lat_col, lon_col, date_col]] = (
            chunk_df[[lat_col, lon_col, date_col]]
            .replace(r"^\s*$", np.nan, regex=True)
        )

        if chunk_df[[lat_col, lon_col, date_col]].isna().any().any():
            missing = chunk_df[[lat_col, lon_col, date_col]].isna().sum()
            raise ValueError(f"Input chunk has missing values: {missing.to_dict()}")

        print(f"\nProcessing rows {chunk_start}..{chunk_end-1} ({len(chunk_df)} rows)")
        try:
            chunk_out = add_gaia_features_v2(
                chunk_df,
                lat_col=lat_col,
                lon_col=lon_col,
                date_col=date_col,
                buffer_m=BUFFER_M,
                scale=SCALE_M
            )
            chunk_out = chunk_out[expected_cols]

            if chunk_out[["Latitude", "Longitude", "Sample Date"]].isna().any().any():
                raise ValueError("Chunk output has blank Latitude/Longitude/Sample Date")

            write_header = (not os.path.exists(out_path)) or (count_rows_in_csv(out_path) == 0)
            chunk_out.to_csv(out_path, mode='a', header=write_header, index=False)

            done_rows = count_rows_in_csv(out_path)
            print(f"Completed rows so far: {done_rows}/{len(df)}")

        except Exception as e:
            print(f"\n❌ Chunk failed at rows {chunk_start}..{chunk_end-1}: {e}")
            print("You can rerun this cell to resume from the last completed chunk.")
            break


extract_chunked(train_df, OUT_TRAIN_PATH, lat_col_train, lon_col_train, date_col_train, "training")
extract_chunked(valid_df, OUT_VALID_PATH, lat_col_valid, lon_col_valid, date_col_valid, "validation")

🚀 Running GAIA extraction for training (chunked)...
Total rows: 9319
Output file: /Users/aaravsonthalia/Projects/Water-Quality-Prediction/Datasets_Ours/gaia_features_training.csv
Chunk size: 200
Resuming from row index: 0

Processing rows 0..199 (200 rows)


Extracting GAIA features: 100%|██████████| 200/200 [00:48<00:00,  4.10it/s]


Completed rows so far: 200/9319

Processing rows 200..399 (200 rows)


Extracting GAIA features: 100%|██████████| 200/200 [00:16<00:00, 11.90it/s]


Completed rows so far: 400/9319

Processing rows 400..599 (200 rows)


...

Completed rows so far: 9200/9319

Processing rows 9200..9318 (119 rows)


Extracting GAIA features: 100%|██████████| 119/119 [00:13<00:00,  8.69it/s]


Completed rows so far: 9319/9319
🚀 Running GAIA extraction for validation (chunked)...
Total rows: 200
Output file: /Users/aaravsonthalia/Projects/Water-Quality-Prediction/Datasets_Ours/gaia_features_validation.csv
Chunk size: 200
Resuming from row index: 0

Processing rows 0..199 (200 rows)


Extracting GAIA features: 100%|██████████| 200/200 [00:11<00:00, 18.08it/s]


Completed rows so far: 200/200
